# Gold template: Type 1 dimension (Databricks)

Builds a dimension that keeps only the current value of each attribute: a change overwrites it.
Use it when nobody needs to report on what a value used to be (a corrected email, a typo in a name).

**Standards this notebook follows**

1. **Surrogate key from Silver.** The dimension's key is the Silver row's `<name>_sk` from the key table, so a
   rebuild never renumbers it and facts never need re-keying. Facts join on this key only.
2. **Unknown member.** Row `-1` always exists, with `Unknown` in its text columns, so a fact whose
   dimension row is missing still joins (inner joins and Power BI relationships keep every fact row).
3. **Business key kept** as an attribute, so people can find a row by the id they know.
4. **Only descriptive attributes.** No measures; numbers you add up belong in a fact.
5. **Upsert, never overwrite.** `MERGE` on the key, updating a row only when its `_row_hash` changed.
   Rows deleted in Silver stay, because facts may still point at them.
6. **Audit columns:** `_row_hash`, `_run_id`, `_loaded_at`.
7. **Layers.** one Unity Catalog catalog per layer: `bronze`, `silver`, `gold`, with keys in `keys`.

## Who supports this notebook

This is an **enterprise template** (`gold-dim-type1` 1.0.0, Databricks). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=gold-dim-type1 platform=databricks version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "silver.crm.customers"
target_table = "gold.sales.dim_customer"
sk_column = "customer_sk"            # the Silver key from the key table; it is this dimension's key
business_key = ["customer_id"]
attributes = {                       # every column of the dimension besides the key, and its type
    "customer_id": "int",
    "name": "string",
    "segment": "string",
    "city": "string",
    "email": "string",
}
run_id = ""          # the pipeline or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=databricks version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
# Job runs pass parameters to a notebook task as widgets; a widget with a parameter's name overrides it.
for _name in ("run_id", "full_reload", "load_date"):
    if _name in globals():
        dbutils.widgets.text(_name, str(globals()[_name]))
        _value = dbutils.widgets.get(_name)
        globals()[_name] = _value.strip().lower() == "true" if isinstance(globals()[_name], bool) else _value

import re
import time
import uuid
from datetime import datetime, timezone
from functools import reduce

from pyspark.errors import AnalysisException
from pyspark.sql import Window
from pyspark.sql import functions as F

UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_TABLE_PART = re.compile(r"[A-Za-z_][A-Za-z0-9_ -]{0,127}")
_COLUMN = re.compile(r"[A-Za-z_][A-Za-z0-9_]{0,127}")
_TYPE = re.compile(r"(?i)(string|boolean|tinyint|smallint|int|bigint|float|double|date|timestamp|decimal\(\d{1,2},\s*\d{1,2}\))")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name with each part in backticks. Parameters arrive from pipelines, so they are checked."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 4 or not all(_TABLE_PART.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(f"`{x}`" for x in parts)


def c(name: str) -> str:
    if not _COLUMN.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return f"`{name}`"


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip()


def table_exists(name: str) -> bool:
    try:
        spark.table(q(name))  # resolved now, so a missing table raises here
        return True
    except AnalysisException:
        return False


def row_hash(columns: list) -> "F.Column":
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    parts = [F.coalesce(F.col(x).cast("string"), F.lit("\u0000")) for x in columns]
    return F.sha2(F.concat_ws("\u001f", *parts), 256)


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"Run {run_id} at {loaded_at:%Y-%m-%d %H:%M:%S} UTC")

## 1. Read Silver

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=databricks version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
for name in business_key:
    if name not in attributes:
        raise ValueError(f"The business key column {name} must be one of the attributes")
attributes = {c_: checked_type(t) for c_, t in attributes.items() if c(c_)}
audit = ["_row_hash", "_run_id", "_loaded_at"]

staged = (spark.table(q(source_table))
          .where(F.col(c(sk_column)) != UNKNOWN_SK)
          .select(F.col(c(sk_column)).cast("bigint").alias(sk_column),
                  *[F.col(c(n)).cast(t).alias(n) for n, t in attributes.items()])
          .withColumn("_row_hash", row_hash(list(attributes)))
          .withColumn("_run_id", F.lit(run_id))
          .withColumn("_loaded_at", F.lit(loaded_at).cast("timestamp")))
staged.createOrReplaceTempView("dim_staged")
def unknown_member_values(types: dict) -> list:
    """The Unknown member's value for each column: 'Unknown' for text, NULL for the rest."""
    return ["'Unknown'" if t.lower() == "string" else f"CAST(NULL AS {t})" for t in types.values()]

## 2. Unknown member, then upsert

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=databricks version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
if not table_exists(target_table):
    ddl = ", ".join([f"{c(sk_column)} BIGINT NOT NULL",
                     *[f"{c(n)} {t}" for n, t in attributes.items()],
                     "`_row_hash` STRING", "`_run_id` STRING", "`_loaded_at` TIMESTAMP"])
    spark.sql(f"CREATE TABLE IF NOT EXISTS {q(target_table)} ({ddl}) USING DELTA CLUSTER BY ({c(sk_column)})")

every = [sk_column, *attributes, *audit]
# The Unknown member, once.
spark.sql(f"""
MERGE INTO {q(target_table)} AS t
USING (SELECT {UNKNOWN_SK} AS {c(sk_column)}, {", ".join(f"{x} AS {c(n)}" for x, n in zip(unknown_member_values(attributes), attributes))},
              'unknown-member' AS _row_hash, '{run_id}' AS _run_id, CAST('{loaded_at:%Y-%m-%d %H:%M:%S}' AS TIMESTAMP) AS _loaded_at) AS s
ON t.{c(sk_column)} = s.{c(sk_column)}
WHEN NOT MATCHED THEN INSERT ({", ".join(c(n) for n in every)}) VALUES ({", ".join(f"s.{c(n)}" for n in every)})
""")
# Every other row: new keys are added, changed rows overwritten.
result = spark.sql(f"""
MERGE INTO {q(target_table)} AS t
USING dim_staged AS s
ON t.{c(sk_column)} = s.{c(sk_column)}
WHEN MATCHED AND t._row_hash <> s._row_hash THEN
    UPDATE SET {", ".join(f"t.{c(n)} = s.{c(n)}" for n in every if n != sk_column)}
WHEN NOT MATCHED THEN
    INSERT ({", ".join(c(n) for n in every)}) VALUES ({", ".join(f"s.{c(n)}" for n in every)})
""").first()
print(f"Dimension MERGE: {result.asDict() if result else 'done'}")

## 3. Check the result

In [ ]:
# kratosforge:standard template=gold-dim-type1 platform=databricks version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
bk = ", ".join(c(k) for k in business_key)
summary = spark.sql(f"""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT {c(sk_column)}) AS key_count,
       SUM(CASE WHEN {c(sk_column)} = {UNKNOWN_SK} THEN 1 ELSE 0 END) AS unknown_members
FROM {q(target_table)}
""").first()
duplicates = spark.sql(f"""
SELECT {bk} FROM {q(target_table)} WHERE {c(sk_column)} <> {UNKNOWN_SK}
GROUP BY {bk} HAVING COUNT(*) > 1 LIMIT 5
""").take(5)
if summary["row_count"] != summary["key_count"] or summary["unknown_members"] != 1 or duplicates:
    raise RuntimeError(f"{target_table} breaks a dimension rule: {summary.asDict()}, duplicate business keys {duplicates}")
print(f"{target_table}: {summary['row_count']} rows including the Unknown member")